In [3]:
# 1. Uninstall broken pre-installs
!pip uninstall -y numpy scikit-learn transformers peft diffusers accelerate torch torchvision

# 2. Install Stable Version Stack
!pip install "numpy<2.0" "scikit-learn==1.3.2" "torch==2.4.1" "torchvision==0.19.1" --index-url https://download.pytorch.org/whl/cu121
!pip install "diffusers==0.30.3" "transformers==4.44.2" "peft==0.12.0" "accelerate==0.34.2" imageio[ffmpeg] tqdm

# IMPORTANT: After this cell finishes, go to 'Run' -> 'Restart Session'

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement scikit-learn==1.3.2 (from versions: none)
ERROR: No matching distribution found for scikit-learn==1.3.2
  Using cached diffusers-0.30.3-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
  Using cached peft-0.12.0-py3-none-any.whl.metadata (13 kB)
  Using cached accelerate-0.34.2-py3-none-any.whl.metadata (19 kB)
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached tokenizers-0.19.1-cp312-cp312

In [2]:
!pip install torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 59.6 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.4 requires scikit-learn, which is not installed.
fastai 2.8.4 requires torch<2.9,>=1.10, but you have torch 2.10.0 which is incompatible.


In [1]:
import torch
import gc
import os
import glob
from PIL import Image
import torch.nn.functional as F
import torchvision.transforms as T
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from peft import LoraConfig, get_peft_model
from tqdm import tqdm
import imageio

# # --- CONFIGURATION ---
BASE_PATH = "/kaggle/input/dataset/datase"
# S1_NAME, S2_NAME = "man", "woman"  # Folder names in your dataset
# S1_TOKEN, S2_TOKEN = "[S1*] candle", "[S2*] teapot"

# # 1. LOAD PIPELINE
def load_training_pipeline():
    gc.collect()
    torch.cuda.empty_cache()
    
    adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
    pipe = AnimateDiffPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", motion_adapter=adapter, torch_dtype=torch.float16, variant="fp16")
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

    # Move UNet to GPU, keep VAE/Text on CPU (float32) for VRAM efficiency
    pipe.unet.to("cuda") 
    pipe.vae.to(dtype=torch.float32)
    pipe.text_encoder.to(dtype=torch.float32)
    pipe.enable_vae_slicing()
    pipe.unet.enable_gradient_checkpointing() 
    return pipe

pipe = load_training_pipeline()

# # 2. PREPARE DISEN-MIX DATA (Section III-C)
# def prepare_data():
#     s1_img = Image.open(glob.glob(os.path.join(BASE_PATH, S1_NAME, "*.jpg"))[0]).convert("RGB").resize((256, 512))
#     s2_img = Image.open(glob.glob(os.path.join(BASE_PATH, S2_NAME, "*.jpg"))[0]).convert("RGB").resize((256, 512))
    
#     # Stitched image (x_mix)
#     x_mix = Image.new('RGB', (512, 512))
#     x_mix.paste(s1_img, (0, 0))
#     x_mix.paste(s2_img, (256, 0))
#     x_mix.save("train_x_mix1.png")
    
#     transform = T.Compose([T.Resize((512, 512)), T.ToTensor(), T.Normalize([0.5], [0.5])])
#     return transform(x_mix).unsqueeze(0)

# train_images = prepare_data()


# --- UPDATE CONFIGURATION ---
# Change these to your new folder names
S1_NAME = "man"    # Folder name
S2_NAME = "woman"  # Folder name 
S1_TOKEN = "[S1*] man"
S2_TOKEN = "[S2*] woman"

def prepare_data():
    # Helper to find any image regardless of extension (.jpg, .png, .JPG, etc.)
    def get_first_image(folder_name):
        search_path = os.path.join(BASE_PATH, folder_name, "*")
        all_files = glob.glob(search_path)
        # Filter for common image extensions
        img_files = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        if not img_files:
            raise FileNotFoundError(f"❌ No images found in folder: {folder_name}. Path checked: {search_path}")
        return img_files[0]

    # Load images using the robust helper
    s1_path = get_first_image(S1_NAME)
    s2_path = get_first_image(S2_NAME)
    
    print(f"✅ Found Subject 1: {s1_path}")
    print(f"✅ Found Subject 2: {s2_path}")

    s1_img = Image.open(s1_path).convert("RGB").resize((256, 512))
    s2_img = Image.open(s2_path).convert("RGB").resize((256, 512))
    
    # Stitched image (x_mix)
    x_mix = Image.new('RGB', (512, 512))
    x_mix.paste(s1_img, (0, 0))
    x_mix.paste(s2_img, (256, 0))
    x_mix.save("train_x_mix1.png")
    
    transform = T.Compose([T.Resize((512, 512)), T.ToTensor(), T.Normalize([0.5], [0.5])])
    return transform(x_mix).unsqueeze(0)

# Run the updated data preparation
train_images = prepare_data()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

2026-02-12 04:01:37.970910: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770868898.147186     203 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770868898.199610     203 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770868898.657062     203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770868898.657111     203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770868898.657115     203 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.82G [00:00<?, ?B/s]

...

unet/diffusion_pytorch_model.fp16.safete(…):   0%|          | 0.00/1.72G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.fp16.safeten(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Found Subject 1: /kaggle/input/dataset/datase/man/03.png
✅ Found Subject 2: /kaggle/input/dataset/datase/woman/03.png


In [2]:
# 1. SETUP LoRA
lora_config = LoraConfig(
    r=8, lora_alpha=32, target_modules=["to_k", "to_v"], 
    lora_dropout=0.05, bias="none"
)
pipe.unet = get_peft_model(pipe.unet, lora_config)

# 2. FREEZE MOTION MODULES (Paper Requirement)
for name, module in pipe.unet.named_modules():
    if "motion_modules" in name:
        for param in module.parameters():
            param.requires_grad = False

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, pipe.unet.parameters()), lr=5e-5)

# 3. TRAINING STEP
def train_step(batch_images, batch_prompts):
    # VAE Encoding (CPU -> GPU)
    with torch.no_grad():
        latents = pipe.vae.encode(batch_images.to("cpu", dtype=torch.float32)).latent_dist.sample()
        latents = (latents * pipe.vae.config.scaling_factor).to("cuda", dtype=torch.float16).unsqueeze(2)

    # Text Encoding (CPU -> GPU)
    inputs = pipe.tokenizer(batch_prompts, padding="max_length", max_length=pipe.tokenizer.model_max_length, truncation=True, return_tensors="pt")
    with torch.no_grad():
        encoder_hidden_states = pipe.text_encoder(inputs.input_ids.to("cpu"))[0].to("cuda", dtype=torch.float16)

    # Denoising
    noise = torch.randn_like(latents)
    timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (latents.shape[0],), device="cuda").long()
    noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
    
    model_pred = pipe.unet(noisy_latents, timesteps, encoder_hidden_states).sample
    return F.mse_loss(model_pred.float(), noise.float())

# 4. RUN LOOP
STITCH_PROMPT = f"a picture is divided into several regions, a {S1_TOKEN} and a {S2_TOKEN}"
print("🚀 Training VideoDreamer...")
for epoch in tqdm(range(50)):
    optimizer.zero_grad()
    loss = train_step(train_images, [STITCH_PROMPT])
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0: print(f"Loss: {loss.item():.4f}")

pipe.unet.save_pretrained("./videodreamer_adapter")

🚀 Training VideoDreamer...


  2%|▏         | 1/50 [00:09<07:30,  9.18s/it]

Loss: 0.0694


 22%|██▏       | 11/50 [01:30<05:19,  8.19s/it]

Loss: 0.0946


...

 62%|██████▏   | 31/50 [04:12<02:33,  8.09s/it]

Loss: 0.7088


 82%|████████▏ | 41/50 [05:33<01:13,  8.14s/it]

Loss: 0.0081


100%|██████████| 50/50 [06:46<00:00,  8.13s/it]


In [3]:
# 1. UNIFY PIPELINE FOR INFERENCE
del optimizer
gc.collect()
torch.cuda.empty_cache()

pipe.to("cuda", dtype=torch.float16)
pipe.unet.eval()

# 2. GENERATE FINAL RESULT
final_prompt = f"A photo of {S1_TOKEN} and {S2_TOKEN} are dancing on a blue table, cinematic, high quality"
print("🎬 Generating Final Video...")

with torch.no_grad():
    generator = torch.Generator(device="cuda").manual_seed(42)
    final_video = pipe(
        final_prompt,
        num_frames=16,
        guidance_scale=12.0,
        num_inference_steps=50,
        generator=generator
    ).frames[0]

imageio.mimsave("videodreamer_candle_teapot.gif", final_video, fps=8)
print("🎉 PROJECT COMPLETE! Download 'videodreamer_candle_teapot.gif' and the adapter folder.")

🎬 Generating Final Video...


  0%|          | 0/25 [00:00<?, ?it/s]

🎉 PROJECT COMPLETE! Download 'videodreamer_candle_teapot.gif' and the adapter folder.


In [8]:
# import ipywidgets as widgets
# from IPython.display import display, clear_output
# import os

# # 1. Scan your Kaggle Input for available subjects
# DATASET_PATH = "/kaggle/input/dreambooth-dataset/DreamBooth_Dataset"
# available_subjects = sorted([f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))])

# # 2. Create UI Elements
# style = {'description_width': 'initial'}

# subject1_dd = widgets.Dropdown(options=available_subjects, value=available_subjects[0], description='Subject 1:', style=style)
# subject2_dd = widgets.Dropdown(options=available_subjects, value=available_subjects[1], description='Subject 2:', style=style)

# prompt_text = widgets.Textarea(
#     value='enter the text',
#     placeholder='Type the action/event here...',
#     description='Action/Prompt:',
#     layout={'height': '80px', 'width': 'auto'},
#     style=style
# )

# gen_button = widgets.Button(
#     description='🚀 Generate VideoDreamer Result',
#     button_style='success', 
#     layout={'width': 'max-content'}
# )

# output_area = widgets.Output()

# # Display the UI
# print("🎨 VideoDreamer Interactive Interface")
# display(widgets.VBox([subject1_dd, subject2_dd, prompt_text, gen_button, output_area]))

import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# 1. SCAN DIRECTORY - Double check this matches your Kaggle sidebar!
# If your dataset is uploaded as 'dataset', use "/kaggle/input/dataset"
# If it's the previous one, use "/kaggle/input/dreambooth-dataset/DreamBooth_Dataset"
# DATASET_PATH = "/kaggle/input/dreambooth-dataset/DreamBooth_Dataset" 

DATASET_PATH = "/kaggle/input/dataset/datase" 

if not os.path.exists(DATASET_PATH):
    # Fallback to check if the path is simply 'dataset'
    DATASET_PATH = "/kaggle/input/dataset"

# Get folders and filter out hidden files
if os.path.exists(DATASET_PATH):
    available_subjects = sorted([f for f in os.listdir(DATASET_PATH) 
                                if os.path.isdir(os.path.join(DATASET_PATH, f))])
else:
    available_subjects = []

# --- SAFETY CHECK ---
if len(available_subjects) == 0:
    print(f"❌ ERROR: No subject folders found in {DATASET_PATH}")
    print("Please check the 'Input' section in your Kaggle sidebar and update DATASET_PATH.")
    # Create a dummy list so the UI doesn't crash during setup
    available_subjects = ["No folders found"]

# 2. CREATE UI ELEMENTS
style = {'description_width': 'initial'}

# Use index 0 for both if only one subject exists to prevent 'out of range'
s1_default = available_subjects[0]
s2_default = available_subjects[1] if len(available_subjects) > 1 else available_subjects[0]

subject1_dd = widgets.Dropdown(options=available_subjects, value=s1_default, description='Subject 1:', style=style)
subject2_dd = widgets.Dropdown(options=available_subjects, value=s2_default, description='Subject 2:', style=style)

prompt_text = widgets.Textarea(
    value='walking together in a school hallway, cinematic',
    placeholder='Type the action/event here...',
    description='Action/Prompt:',
    layout={'height': '80px', 'width': '400px'},
    style=style
)

gen_button = widgets.Button(
    description='🚀 Generate VideoDreamer Result',
    button_style='success', 
    layout={'width': 'max-content'}
)

output_area = widgets.Output()

# Display the UI
print("🎨 VideoDreamer Interactive Interface")
display(widgets.VBox([subject1_dd, subject2_dd, prompt_text, gen_button, output_area]))

🎨 VideoDreamer Interactive Interface


In [11]:
import torch
import gc
import imageio
from IPython.display import Image as DisplayImage, clear_output

# 1. PREPARE MODEL FOR INFERENCE (One-time setup for the button)
def prepare_for_demo():
    gc.collect()
    torch.cuda.empty_cache()
    # Force everything to GPU and ensure VAE is safe (prevents black screen)
    pipe.to("cuda", dtype=torch.float16)
    pipe.vae.to(dtype=torch.float32) # Hybrid precision for stability
    pipe.unet.eval()
    pipe.enable_vae_slicing()
    print("✅ Model Unified on GPU. UI is now active.")

prepare_for_demo()

# 2. DEFINE WHAT HAPPENS WHEN YOU CLICK THE BUTTON
def on_button_clicked(b):
    with output_area:
        # Clear previous video/text from the UI
        clear_output()
        
        # Get values from the dropdowns and text box
        s1 = subject1_dd.value
        s2 = subject2_dd.value
        user_action = prompt_text.value
        
        # The Paper's Token Logic
        final_prompt = f"A photo of [S1*] {s1} and [S2*] {s2} {user_action}, cinematic, 4k"
        
        print(f"⌛ Generating: '{final_prompt}'")
        print("🚀 Status: Processing 16 frames on Tesla T4. Please wait ~2 mins...")

        try:
            with torch.no_grad():
                # Use CUDA generator for speed
                generator = torch.Generator(device="cuda").manual_seed(42)
                
                # Execute Generation
                with torch.autocast("cuda"):
                    output = pipe(
                        final_prompt,
                        num_frames=16,
                        height=512,
                        width=512,
                        guidance_scale=8.5,
                        num_inference_steps=25,
                        generator=generator
                    ).frames[0]
            
            # Save the GIF
            fn = f"videodreamer_{s1}_{s2}.gif"
            imageio.mimsave(fn, output, fps=8)
            
            # Show the final result in the UI
            print(f"✅ Success! Video saved as {fn}")
            display(DisplayImage(data=open(fn, 'rb').read(), format='png'))
            
        except Exception as e:
            print(f"❌ Error during generation: {e}")
            print("Tip: If you see 'Out of Memory', please Restart Session and run the setup cells.")

# 3. LINK THE BUTTON TO THE LOGIC
gen_button.on_click(on_button_clicked)

print("🚀 System Ready! Select subjects above and click the Green Button.")

✅ Model Unified on GPU. UI is now active.
🚀 System Ready! Select subjects above and click the Green Button.
